# Remote↔Central perfSONAR Mesh: Automated Setup & Archiving

This notebook provisions a **central VM** and multiple **remote VMs**, installs perfSONAR components on each, wires up **pSConfig** for test orchestration, and configures **local and optional central archiving** so results appear in Grafana.

## What this notebook does

* **Provisioning**

  * Creates one **central VM** and N **remote VMs**.
  * Installs on **all VMs**: `perfsonar-testpoint`, `perfsonar-archive`, and `perfsonar-grafana`.
* **Configuration**

  * **Central** publishes a **pSConfig** template describing all hosts and archives.
  * **Remote Nodes** subscribe to the central template and also load a **local pSConfig** (self + central).
* **Measurements**

  * Schedules the following tests **in both directions** (A→B and B→A where applicable):

    * **Throughput:** `iperf3` (e.g., `-P 4 -t 60 -i 10 -O 10`)
    * **Latency:** `owping`, `twping`, `halfping`
    * **RTT:** `ping`, `tcpping`
    * **MTU:** `fwmtu`
    * **Clock:** `psclock`
    * **Trace:** `traceroute`
  * Test cadences are parameterized; defaults can be set at the top of the notebook.
* **Archiving**

  * **Local archive on each remote VM** via HTTP archiver → Logstash → OpenSearch.
  * **Optional central archiving** (mirrored submission) for fleet-wide dashboards.
* **Visualization**

  * Each VM runs Grafana with perfSONAR dashboards (OpenSearch or pSConfig-driven views).

## Key parameters (set at the top)

* `TOTAL_NODE_CNT = 3`. Controls the number of nodes/VMs to use for the experiment `1(Central) + (N-1) (Remote)`.
* `TEST_INTERVAL = 10M`. Controls the test interval with allowed values `10M`, `2H`, `4H` and `6H`.
* **Archive behavior:**
  * Always send to **local** archive on each remote server.
  * Optionally **mirror** to central by enabling `USE_CENTRAL_ARCHIVE = True`

![](./images/perfsonar-psconfig.png)

## Import the FABlib Library

FABlib is used to programmatically create and manage FABRIC resources such as slices and VMs.

In [1]:
from ipaddress import ip_address, IPv4Address, IPv6Address, IPv4Network, IPv6Network
import ipaddress, os, json

from fabrictestbed_extensions.fablib.fablib import FablibManager as fablib_manager

fablib = fablib_manager()
                     
fablib.show_config();

Credential Manager,cm.fabric-testbed.net
Orchestrator,orchestrator.fabric-testbed.net
Project ID,a70de2f5-9e12-4b6b-b412-0ae1a2c553b0
Token File,/home/fabric/.tokens.json
Bastion Host,bastion.fabric-testbed.net
Bastion Username,ranna01_0000554580
Bastion Private Key File,/home/fabric/work/fabric_config/fabric_bastion_key
Slice Private Key File,/home/fabric/work/fabric_config/slice_key
Slice Public Key File,/home/fabric/work/fabric_config/slice_key.pub
Log File,/tmp/fablib/fablib.log
Log Level,INFO


## Create the Experiment Slice

This step provisions the VMs on the FABRIC testbed.

In [2]:
slice_name = 'compass-perfSonar-fabric-multi-site'

TOTAL_NODE_CNT = 2

# sites = fablib.get_random_sites(TOTAL_NODE_CNT)
# print(f"Sites: {sites}")

central_node_name = "central"
remote_node_name = "remote"

USE_CENTRAL_ARCHIVE = True
TEST_INTERVAL = '10M' # Allowed values 10M, 2H, 4H, 6H

In [3]:
# sites = fablib.get_random_sites(TOTAL_NODE_CNT)
sites = ["SEAT", "DALL"]
print(f"Sites: {sites}")

Sites: ['SEAT', 'DALL']


In [4]:
# Create Slice
slice = fablib.new_slice(name=slice_name)

for i, s in enumerate(sites):
    prefix = central_node_name if i == 0 else remote_node_name

    # Use regular FabNetv4 instead of IPv4Ext
    net = slice.add_l3network(
        name=f"{s}-l3",
        type='IPv4'
    )

    node = slice.add_node(
        name=f"{prefix}-{s}",
        site=s,
        image="default_ubuntu_24",
        cores=16,
        ram=32,
        disk=100
    )

    iface = node.add_component(
        model='NIC_Basic',
        name='nic1'
    ).get_interfaces()[0]

    # Automatically configure FabNetv4 IP
    iface.set_mode('auto')
    net.add_interface(iface)

    # Route between FABRIC FabNetv4 networks
    node.add_route(
        subnet=fablib.FABNETV4_SUBNET,
        next_hop=net.get_gateway()
    )

# Submit Slice Request
slice.submit()


Retry: 11, Time: 257 sec


ID,09da91c8-4eac-4a30-8b0a-0c67eefba57d
Name,compass-perfSonar-fabric-multi-site
Lease Expiration (UTC),2026-09-18 01:42:14 +0000
Lease Start (UTC),2026-09-17 01:42:14 +0000
Project ID,a70de2f5-9e12-4b6b-b412-0ae1a2c553b0
State,StableOK
Email,ranna01@jaguar.tamu.edu
UserId,e335f96a-332c-49ec-9810-0d6cbe323c83


ID,Name,Cores,RAM,Disk,Image,Image Type,Host,Site,Username,Management IP,State,Error,SSH Command,Public SSH Key File,Private SSH Key File
ce89e5c6-b04c-49a3-a04c-974f944f25da,central-SEAT,16,32,100,default_ubuntu_24,qcow2,seat-w2.fabric-testbed.net,SEAT,ubuntu,2001:400:a100:3080:f816:3eff:fe94:2f10,Active,,ssh -i /home/fabric/work/fabric_config/slice_key -F /home/fabric/work/fabric_config/ssh_config ubuntu@2001:400:a100:3080:f816:3eff:fe94:2f10,/home/fabric/work/fabric_config/slice_key.pub,/home/fabric/work/fabric_config/slice_key
714bd6a7-5bc7-4998-a0b6-07ceb48e84e0,remote-DALL,16,32,100,default_ubuntu_24,qcow2,dall-w1.fabric-testbed.net,DALL,ubuntu,2001:400:a100:3000:f816:3eff:febc:fd55,Active,,ssh -i /home/fabric/work/fabric_config/slice_key -F /home/fabric/work/fabric_config/ssh_config ubuntu@2001:400:a100:3000:f816:3eff:febc:fd55,/home/fabric/work/fabric_config/slice_key.pub,/home/fabric/work/fabric_config/slice_key


ID,Name,Layer,Type,Site,Gateway,Subnet,State,Error
4e8bf353-dfbc-4c5b-a89f-ecd101649b27,DALL-l3,L3,FABNetv4,DALL,10.133.131.1,10.133.131.0/24,Active,
a27a7203-e825-45dc-a6ec-bbf29141233e,SEAT-l3,L3,FABNetv4,SEAT,10.139.10.1,10.139.10.0/24,Active,


Name,Short Name,Node,Network,Bandwidth,VLAN,MAC,Physical Device,Device,Mode,IP Address,Numa Node,Switch Port
central-SEAT-nic1-p1,p1,central-SEAT,SEAT-l3,100,,1A:72:3E:D2:AE:B2,enp7s0,enp7s0,auto,10.139.10.2,4,HundredGigE0/0/0/7
remote-DALL-nic1-p1,p1,remote-DALL,DALL-l3,100,,0E:7B:46:7C:EC:BB,enp7s0,enp7s0,auto,10.133.131.2,6,HundredGigE0/0/0/5



Time to print interfaces 257 seconds


'09da91c8-4eac-4a30-8b0a-0c67eefba57d'

## Configure `/etc/hosts`

For consistent name resolution across the fleet, update `/etc/hosts` on **every** VM with the same mappings.

**Steps**

* Add an entry for each node: IP address followed by its hostname (and optional aliases).
* Keep the file identical on all nodes to prevent lookup mismatches during monitoring/orchestration.

**Example**

```
10.128.1.2    central-STAR
10.128.127.2  remote-RUTG
10.128.254.2  remote-NEWY
```

In [5]:
etc_hosts = ""
for server in slice.get_nodes():
    print(f"node: {server.get_name()}")
    server_ip=server.get_interface(network_name=f"{server.get_site()}-l3").get_ip_addr()
    etc_hosts += f"{server_ip} {server.get_name()}\n"

for server in slice.get_nodes():
    server.execute(f'sudo sh -c \'echo "{etc_hosts}" >> /etc/hosts\'')

In [6]:
for server in slice.get_nodes():
    print(f"/etc/hosts on {server.get_name()}")
    server.execute('cat /etc/hosts')

/etc/hosts on central-SEAT
# Your system has configured 'manage_etc_hosts' as True.
# As a result, if you wish for changes to this file to persist
# then you will need to either
# a.) make changes to the master file in /etc/cloud/templates/hosts.debian.tmpl
# b.) change or remove the value of 'manage_etc_hosts' in
#     /etc/cloud/cloud.cfg or cloud-config from user-data
#
127.0.1.1 ce89e5c6-b04c-49a3-a04c-974f944f25da-central-seat.novalocal central-seat
127.0.0.1 localhost

# The following lines are desirable for IPv6 capable hosts
::1 localhost ip6-localhost ip6-loopback
ff02::1 ip6-allnodes
ff02::2 ip6-allrouters

10.139.10.2 central-SEAT
10.133.131.2 remote-DALL

/etc/hosts on remote-DALL
# Your system has configured 'manage_etc_hosts' as True.
# As a result, if you wish for changes to this file to persist
# then you will need to either
# a.) make changes to the master file in /etc/cloud/templates/hosts.debian.tmpl
# b.) change or remove the value of 'manage_etc_hosts' in
#     /et

## Clone the repository

In [7]:
slice = fablib.get_slice(slice_name)

for server in slice.get_nodes():
    stdout, stderr = server.execute("git clone https://github.com/kthare10/perfsonar-extensions.git /home/ubuntu/perfsonar-extensions")

Cloning into '/home/ubuntu/perfsonar-extensions'...
Cloning into '/home/ubuntu/perfsonar-extensions'...


## Install perfSONAR stack (Testpoint, Archive, Grafana)

Install the full perfSONAR stack on **every** VM—both central and remote—for a consistent setup:

* **Testpoint**: pScheduler and measurement tools.
* **Archive**: Logstash/OpenSearch pipeline for storing results.
* **Grafana**: Dashboards for visualization.

This ensures each node can run tests, store results locally, and (optionally) mirror data to the central archive.

In [8]:
ip_map = {}
central_host = ""
for n in slice.get_nodes():
    n.upload_directory('node_tools','.')
    ip_addr = n.get_interface(network_name=f"{n.get_site()}-l3").get_ip_addr() 
    ip_map[n.get_name()] = ip_addr
    if central_node_name in n.get_name():
        central_host = n.get_name()
    n.execute('cd perfsonar-extensions/native/ && sudo scripts/perfsonar-install.sh', quiet=True, output_file=f"{n.get_name()}.log")

In [9]:
for n in slice.get_nodes():
    print("\n==============================")
    print(n.get_name())
    print("==============================")

    n.execute("""
        echo "pscheduler:"
        command -v pscheduler || echo "MISSING"

        echo "psconfig:"
        command -v psconfig || echo "MISSING"

        echo "logstash apache config:"
        if [ -f /etc/apache2/conf-available/apache-logstash.conf ]; then
            echo "FOUND"
        else
            echo "MISSING"
        fi
    """)


central-SEAT
pscheduler:
/usr/bin/pscheduler
psconfig:
/usr/bin/psconfig
logstash apache config:
FOUND

remote-DALL
pscheduler:
/usr/bin/pscheduler
psconfig:
/usr/bin/psconfig
logstash apache config:
FOUND


In [10]:
for n in slice.get_nodes():
    logfile = f"{n.get_name()}.log"
    print("\n==========", logfile, "==========")
    with open(logfile, "r", errors="ignore") as f:
        print("".join(f.readlines()[-30:]))


========== central-SEAT.log ==========
apache2_invoke ssl: already enabled
apache2_invoke proxy: already enabled
apache2_invoke proxy_http: already enabled
Setting up perfsonar-grafana-toolkit (5.2.6-1) ...
Processing triggers for man-db (2.12.0-4build2) ...
debconf: unable to initialize frontend: Dialog
debconf: (Dialog frontend will not work on a dumb terminal, an emacs shell buffer, or without a controlling terminal.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype

Running kernel seems to be up-to-date.

Restarting services...

Service restarts being deferred:
 /etc/needrestart/restart.d/dbus.service
 systemctl restart getty@tty1.service
 systemctl restart serial-getty@ttyS0.service
 systemctl restart systemd-logind.service
 systemctl restart unattended-upgrades.service

No containers need to be restarted.

User sessions running outdated bin

## Archive access control — update Apache `/logstash` allowlist & auth

Lock down the Logstash ingest endpoint so only approved hosts (and/or authenticated clients) can post results.

**Steps**

1. **Edit the `/logstash` location block**
   File: `/etc/apache2/conf-available/apache-logstash.conf`

```apache
# ... existing ProxyPass/ProxyPassReverse for /logstash ...

<Location "/logstash">
  AuthType Basic
  AuthName "Logstash Ingest"
  # User file created by the archive install (adjust if different)
  AuthUserFile /etc/perfsonar/opensearch/logstash_login

  # Allow either a valid user OR specific IPs (including localhost)
  <RequireAny>
    Require valid-user
    Require ip 127.0.0.1 ::1
    # Add remote testpoints here (examples):
    Require ip 10.128.127.0/24
    Require ip 203.0.113.45
  </RequireAny>
</Location>
```

3. **Validate & reload Apache**

```bash
sudo apachectl -t
sudo systemctl reload apache2
```

4. **Quick checks**

* On the archive host: `sudo tail -f /var/log/apache2/access.log /var/log/apache2/error.log`
* From a remote VM, run a test that uses the HTTP archiver, then on the archive:

  ```bash
  pscheduler archiving-summary PT30M
  ```

  You should see successes increase.

**Notes**

* Keep HTTPS enabled for `/logstash` and rotate credentials as needed.
* Using both `Require valid-user` and `Require ip` lets you mix “trusted by IP” remote VMs with authenticated clients.

In [11]:
all_ips = ip_map.values()
for n in slice.get_nodes():
    n.execute(f'cd perfsonar-extensions/native/ && sudo scripts/allow_logstash_ips.sh  {" ".join(map(str, all_ips))}')

Backing up /etc/apache2/conf-available/apache-logstash.conf -> /etc/apache2/conf-available/apache-logstash.conf.bak.20260917-021211
Adding: Require ip 10.139.10.2
Adding: Require ip 10.133.131.2
Validating Apache config...
[Thu Sep 17 02:12:11.247752 2026] [proxy:warn] [pid 48152:tid 137042667984768] AH01146: Ignoring parameter 'status=+I' for worker 'http://localhost:3000' because of worker sharing
Syntax OK
Reloading Apache...
Done. Updated /etc/apache2/conf-available/apache-logstash.conf
Backing up /etc/apache2/conf-available/apache-logstash.conf -> /etc/apache2/conf-available/apache-logstash.conf.bak.20260917-021211
Adding: Require ip 10.139.10.2
Adding: Require ip 10.133.131.2
Validating Apache config...
[Thu Sep 17 02:12:11.597112 2026] [proxy:warn] [pid 47914:tid 126994155992960] AH01146: Ignoring parameter 'status=+I' for worker 'http://localhost:3000' because of worker sharing
Syntax OK
Reloading Apache...
Done. Updated /etc/apache2/conf-available/apache-logstash.conf


In [12]:
all_hosts = [x for name, ip in ip_map.items() for x in (name, str(ip))]
print(all_hosts)

['central-SEAT', '10.139.10.2', 'remote-DALL', '10.133.131.2']


## pSConfig

Orchestrate tests with a central template and have remote VMs subscribe to it.

### Central VM: build & publish the mesh template

1. Create a JSON template defining **addresses, groups, tests, schedules, and archives** (the HTTP `/logstash` archive) using `psconfig/psconfig_builder.py`.
2. Publish it so remote VMs can subscribe:

   ```bash
   sudo psconfig publish psconfig/psconfig.json
   sudo psconfig remote add https://localhost/psconfig/psconfig.json
   ```

### Remote VMs: subscribe (and auto-configure archives)

1. Create a JSON template defining **addresses, groups, tests, schedules, and archives** (the HTTP `/logstash` archive) using `psconfig/psconfig_builder.py`.
2. Add a small **local** template on each remote VM (self + central) for specific extras, and subscribe to it as a file:

   ```bash
   sudo psconfig publish psconfig/psconfig.json
   sudo psconfig remote add https://localhost/psconfig/psconfig.json
   ```

### Verify

* After \~1 minute, tasks should appear:

  ```bash
  pscheduler schedule
  pscheduler archiving-summary PT30M
  ```
* In Grafana (central and/or remote), confirm data is arriving for the time window covering the first runs.


In [13]:
for n in slice.get_nodes():
    if central_host in n.get_name():
        cmd = (
                f'cd perfsonar-extensions/native/ && python3 psconfig/psconfig_builder.py --base_config_file psconfig/base_psconfig.json '
                f'--output_file psconfig/psconfig.json --no_add_tests --host_list {" ".join(map(str, all_hosts))}'
        )
        print(cmd)
        stdout, stderr = n.execute(cmd)
    else:
        if USE_CENTRAL_ARCHIVE:
            cmd = (
                    f"cd perfsonar-extensions/native/ && python3 psconfig/psconfig_builder.py "
                    f"--base_config_file psconfig/base_psconfig.json "
                    f"--output_file psconfig/psconfig.json "
                    f"--remote {ip_map[central_host]} "
                    f'--host_list {n.get_name()} {ip_map[n.get_name()]} {central_host} {ip_map[central_host]}'
                )
        else:
            cmd = (
                    f"cd perfsonar-extensions/native/ && python3 psconfig/psconfig_builder.py "
                    f"--base_config_file psconfig/base_psconfig.json "
                    f"--output_file psconfig/psconfig.json "
                    f'--host_list {n.get_name()} {ip_map[n.get_name()]} {central_host} {ip_map[central_host]}'
                )
            
        print(cmd)
        stdout, stderr = n.execute(cmd)

    stdout, stderr = n.execute('cd perfsonar-extensions/native/ && sudo psconfig validate psconfig/psconfig.json')
    stdout, stderr = n.execute('cd perfsonar-extensions/native/ && sudo psconfig publish psconfig/psconfig.json')

    stdout, stderr = n.execute('sudo psconfig remote add "https://localhost/psconfig/psconfig.json"')
    print("==============================================================================================================================================")
    print()

cd perfsonar-extensions/native/ && python3 psconfig/psconfig_builder.py --base_config_file psconfig/base_psconfig.json --output_file psconfig/psconfig.json --no_add_tests --host_list central-SEAT 10.139.10.2 remote-DALL 10.133.131.2
/usr/lib/python3/dist-packages/psconfig/client/psconfig/parsers/task_generator.py:244: SyntaxWarning: invalid escape sequence '\d'
  test_data_json = re.sub("['\"](\d+)['\"]", r'\1',test_data_json) #removes both single and double quotes
Loading config ...... OK
Validating JSON schema ...... OK
Verifying object references ...... OK
pScheduler Validation (Quick) ...... OK

pSConfig JSON is valid
                                               
Success! File saved to /usr/lib/perfsonar/web-psconfig/psconfig.json
Published file can be accessed at https://central-SEAT/psconfig/psconfig.json
Execute the following on a host running an agent to use this file:
    psconfig remote add "https://central-SEAT/psconfig/psconfig.json"

=== pScheduler Agent ===
Added remote

In [14]:
for n in slice.get_nodes():
    print("\n==============================")
    print(n.get_name())
    print("==============================")

    n.execute("""
        echo "=== pScheduler schedule ==="
        pscheduler schedule

        echo
        echo "=== Archiving summary ==="
        pscheduler archiving-summary PT30M
    """)


central-SEAT
=== pScheduler schedule ===
2026-09-17T02:21:05+00:00 - 2026-09-17T02:22:24+00:00 (Pending)
throughput --source 10.133.131.2 --dest 10.139.10.2 --duration PT60S --interval PT10S --omit PT10S (Run with tool 'iperf3')
https://central-seat/pscheduler/tasks/2a5add28-1137-4e1d-8166-d3d0cf87846e/runs/91605d0c-62b1-44bf-928b-da565e31a3d3

=== Archiving summary ===
Nothing archived since 2026-09-17 01:45:26.886257+00:00

remote-DALL
=== pScheduler schedule ===
2026-09-17T02:21:05+00:00 - 2026-09-17T02:22:24+00:00 (Pending)
throughput --source 10.133.131.2 --dest 10.139.10.2 --duration PT60S --interval PT10S --omit PT10S (Run with tool 'iperf3')
https://remote-dall/pscheduler/tasks/2a5add28-1137-4e1d-8166-d3d0cf87846e/runs/91605d0c-62b1-44bf-928b-da565e31a3d3

=== Archiving summary ===
Nothing archived since 2026-09-17 01:45:27.577116+00:00


In [15]:
for n in slice.get_nodes():
    print("\n==============================")
    print(n.get_name())
    print("==============================")

    n.execute("""
        pscheduler archiving-summary PT1H
    """)


central-SEAT

   Succeeded          Failed        Label
----------------  ----------------  ----
       28 (100%)          0 (  0%)  http
----------------  ----------------  ----
       28 (100%)          0 (  0%)  Total (28)


remote-DALL

   Succeeded          Failed        Label
----------------  ----------------  ----
       40 (100%)          0 (  0%)  http
----------------  ----------------  ----
       40 (100%)          0 (  0%)  Total (40)



## Grafana Access (FabNetv4Ext) – Requires Node Connection to FabNetv4Ext

Grafana can be accessed via the FabNetv4Ext IP address assigned to each node. To connect from your local network, make sure to configure routing first:

* Add a route to the FabNetv4Ext network.
* After the route is added, open Grafana using the node’s FabNetv4Ext IP.
* **Note**: The subnet `152.54.8.0/23` shown in the example below is for illustration only. Update it to match your specific configuration.


In [ ]:
# for n in slice.get_nodes():
#     network_name=f"{n.get_site()}-l3"
#     nw = slice.get_network(network_name)
#     n.execute(f'sudo ip route add 152.54.8.0/23 via {nw.get_gateway()}')

In [ ]:
# for iface in slice.get_interfaces():
#     print(f"Access grafana on {iface.get_node().get_name()} via https://{iface.get_ip_addr()}/grafana")

## Grafana Access (SSH Tunnels via Management Network)

Use an SSH local port forward so you can open the remote Grafana UI in your local browser.

In [16]:
fablib.create_ssh_tunnel_config(overwrite=True)


SSH tunnel config created and zipped at: /home/fabric/work/fabric_config/fabric_ssh_tunnel_tools.tgz

Download Instructions:
Download your custom `fabric_ssh_tunnel_tools.tgz` file from the `fabric_config` folder.

Usage Instructions:
1. Unzip the archive and place the resulting `fabric_ssh_tunnel_tools/` folder somewhere accessible from your terminal.
2. Open a terminal window (on Windows, use PowerShell).
3. Use `cd` to navigate into the `fabric_ssh_tunnel_tools` folder.
4. In your terminal, run the SSH tunnel command generated by the next notebook cell.
    


In [4]:
slice_name = 'compass-perfSonar-fabric-multi-site'
slice = fablib.get_slice(name=slice_name)

print(slice.get_name())
print(slice.get_state())

User: ranna01@jaguar.tamu.edu bastion key is valid!
Configuration is valid
compass-perfSonar-fabric-multi-site
StableOK


## perfSONAR Toolkit Grafana

Use SSH local port forwards to map your **local** ports to each remote host’s **HTTPS (443)**. Then open the URLs below in your browser.

### Central Node (maps local 8443 → remote 443)

```bash
ssh -N -L 8443:localhost:443 <USER>@<CENTRAL_IP>
# Browse: https://127.0.0.1:8443
```

### Remote VMs (add one tunnel per VM)

```bash
# Remote-1  (local 8444 → remote-1:443)
ssh -N -L 8444:localhost:443 <USER>@<REMOTE1_IP>
# Browse: https://127.0.0.1:8444

# Remote-2  (local 8445 → remote-2:443)
ssh -N -L 8445:localhost:443 <USER>@<REMOTE2_IP>
# Browse: https://127.0.0.1:8445
```

> Tip: add `-f` to send the tunnel to the background (`ssh -fN -L ...`).
> If remote VMs are only reachable via central, use a jump host:
> `ssh -J <USER>@<CENTRAL_IP> -N -L 8444:localhost:443 <USER>@REMOTE1_IP>`

### Notes

* perfSONAR/Grafana often uses a self-signed cert — your browser may warn; proceed if you trust the host.
* Log in with your Grafana credentials.
* Stop a tunnel with `Ctrl+C` (or kill the background `ssh` if you used `-f`).


In [5]:
# Port on your local machine that you want to map the File Browser to.
# local_port=8443
local_port=8463
# Local interface to map the File Browser to (can be `localhost`)
local_host='127.0.0.1'

# Port on the node used by the File Browser Service
target_port='443'

for n in slice.get_nodes():
    # Username/node on FABRIC
    target_host=f'{n.get_username()}@{n.get_management_ip()}'
    print(f'Tunnel command for {n.get_name()}')
    print(f'ssh  -L {local_host}:{local_port}:127.0.0.1:{target_port} -i {os.path.basename(fablib.get_default_slice_public_key_file())[:-4]} -F ssh_config {target_host}')
    local_port += 1
    print()


Tunnel command for central-SEAT
ssh  -L 127.0.0.1:8463:127.0.0.1:443 -i slice_key -F ssh_config ubuntu@2001:400:a100:3080:f816:3eff:fe94:2f10

Tunnel command for remote-DALL
ssh  -L 127.0.0.1:8464:127.0.0.1:443 -i slice_key -F ssh_config ubuntu@2001:400:a100:3000:f816:3eff:febc:fd55



## Verification

### pScheduler health

```bash
# Basic end-to-end checks (network, tools, auth, archives)
pscheduler troubleshoot

# What’s scheduled / running
pscheduler schedule
pscheduler task --state on-run

# Archiving success/fail in the last 30 minutes
pscheduler archiving-summary PT30M
```

### OpenSearch spot-checks (recent docs)

```bash
# Replace with your archive host and credentials
curl -k -u '<user>:<pass>' \
  'https://<archive-host>/opensearch/pscheduler*/_search?size=5' \
  -H 'Content-Type: application/json' -d '{
    "sort": [{"pscheduler.start_time":{"order":"desc"}}]
  }' | jq '.[].hits.hits[]._source
           | { time: .pscheduler.start_time,
               type: .test.type,
               src: .test.spec.source,
               dst: .test.spec.dest,
               summary: .summary }'
```

You should see `src` and `dst` populated; if blank, ensure tests specify **both** `--source` and `--dest` (pSConfig does this automatically).

### Grafana dashboards

* Open the dashboard and set the time range to include your latest runs.
* Confirm panels populate and show **source/dest** labels.
* If no data:

  * Check the Grafana data source (OpenSearch URL, index `pscheduler*`, time field `pscheduler.start_time`).
  * Re-check `pscheduler archiving-summary` and the archive’s Apache/Logstash logs.


# Automated Archive Analysis and Cross-Tool Data Extraction
This section validates OpenSearch access, summarizes the DALL→SEAT archive, implements the four automated analysis extensions, and shows how the perfSONAR measurements used by the cross-tool case study were extracted.


In [10]:
# ============================================================
# NOTEBOOK 3 - Verify node and OpenSearch access
# ============================================================

central3 = slice.get_node(name="central-SEAT")

stdout, stderr = central3.execute(r"""
echo "=== NOTEBOOK 3 CHECK ==="

echo -n "OpenSearch service: "
sudo systemctl is-active opensearch 2>/dev/null || true

echo -n "Login file exists: "
sudo test -f /etc/perfsonar/opensearch/opensearch_login \
  && echo "YES" \
  || echo "NO"

echo
echo "Auth test:"

sudo bash -c '
read -r OS_USER OS_PASS < /etc/perfsonar/opensearch/opensearch_login

CODE=$(curl -ks \
    -o /dev/null \
    -w "%{http_code}" \
    -u "${OS_USER}:${OS_PASS}" \
    https://localhost/opensearch/pscheduler/_search)

echo "HTTP status: $CODE"
'
""")

print(stdout)

if stderr:
    print(stderr)

=== NOTEBOOK 3 CHECK ===
OpenSearch service: active
Login file exists: YES

Auth test:
HTTP status: 200
=== NOTEBOOK 3 CHECK ===
OpenSearch service: active
Login file exists: YES

Auth test:
HTTP status: 200



In [5]:
# ============================================================
# CONTRIBUTION TASK 1
# Automatic Full-Archive Site-to-Site Performance Analysis
# Notebook 3: remote-DALL -> central-SEAT
# ============================================================

import json
import base64

central3 = slice.get_node(name="central-SEAT")


def query_opensearch_nb3(query_body):

    body_json = json.dumps(query_body)

    body_b64 = base64.b64encode(
        body_json.encode("utf-8")
    ).decode("utf-8")

    cmd = f"""
sudo bash -c '
read -r OS_USER OS_PASS < /etc/perfsonar/opensearch/opensearch_login

echo "{body_b64}" | base64 -d | \
curl -ks \
  -u "${{OS_USER}}:${{OS_PASS}}" \
  -H "Content-Type: application/json" \
  --data-binary @- \
  https://localhost/opensearch/pscheduler/_search
'
"""

    stdout, stderr = central3.execute(cmd)

    data = json.loads(stdout)

    if "error" in data:
        raise RuntimeError(
            "OpenSearch error: " +
            json.dumps(data["error"], indent=2)
        )

    return data


# ============================================================
# 1. THROUGHPUT
# ============================================================

throughput_query = {
    "size": 0,
    "track_total_hits": True,
    "query": {
        "bool": {
            "must": [
                {"match": {"test.type": "throughput"}},
                {"term": {"result.succeeded": True}}
            ]
        }
    },
    "aggs": {
        "avg_value": {
            "avg": {"field": "result.throughput"}
        },
        "min_value": {
            "min": {"field": "result.throughput"}
        },
        "max_value": {
            "max": {"field": "result.throughput"}
        }
    }
}

t3_data = query_opensearch_nb3(throughput_query)

throughput3_count = t3_data["hits"]["total"]["value"]

throughput3_avg = (
    t3_data["aggregations"]["avg_value"]["value"]
    / 1_000_000
)

throughput3_min = (
    t3_data["aggregations"]["min_value"]["value"]
    / 1_000_000
)

throughput3_max = (
    t3_data["aggregations"]["max_value"]["value"]
    / 1_000_000
)


# ============================================================
# 2. RTT + PACKET LOSS
# ============================================================

rtt_query = {
    "size": 0,
    "track_total_hits": True,
    "query": {
        "bool": {
            "must": [
                {"match": {"test.type": "rtt"}},
                {"term": {"result.succeeded": True}}
            ]
        }
    },
    "aggs": {
        "avg_rtt": {
            "avg": {"field": "result.rtt.mean"}
        },
        "min_rtt": {
            "min": {"field": "result.rtt.mean"}
        },
        "max_rtt": {
            "max": {"field": "result.rtt.mean"}
        },
        "avg_loss": {
            "avg": {"field": "result.packets.loss"}
        }
    }
}

r3_data = query_opensearch_nb3(rtt_query)

rtt3_count = r3_data["hits"]["total"]["value"]

rtt3_avg = (
    r3_data["aggregations"]["avg_rtt"]["value"]
    * 1000
)

rtt3_min = (
    r3_data["aggregations"]["min_rtt"]["value"]
    * 1000
)

rtt3_max = (
    r3_data["aggregations"]["max_rtt"]["value"]
    * 1000
)

rtt3_loss = (
    r3_data["aggregations"]["avg_loss"]["value"]
    * 100
)


# ============================================================
# 3. LATENCY + JITTER + PACKET LOSS
# ============================================================

latency_query = {
    "size": 0,
    "track_total_hits": True,
    "query": {
        "bool": {
            "must": [
                {"match": {"test.type": "latencybg"}},
                {"term": {"result.succeeded": True}}
            ]
        }
    },
    "aggs": {
        "avg_latency": {
            "avg": {"field": "result.latency.mean"}
        },
        "avg_jitter": {
            "avg": {"field": "result.latency.stddev"}
        },
        "avg_loss": {
            "avg": {"field": "result.packets.loss"}
        }
    }
}

l3_data = query_opensearch_nb3(latency_query)

latency3_count = l3_data["hits"]["total"]["value"]

latency3_avg = (
    l3_data["aggregations"]["avg_latency"]["value"]
)

jitter3_avg = (
    l3_data["aggregations"]["avg_jitter"]["value"]
)

latency3_loss = (
    l3_data["aggregations"]["avg_loss"]["value"]
    * 100
)


# ============================================================
# CLEAN OUTPUT
# ============================================================

print()
print("=" * 64)
print("FULL ARCHIVE AUTOMATIC SITE-TO-SITE PERFORMANCE ANALYSIS")
print("Notebook 3: remote-DALL -> central-SEAT")
print("=" * 64)

print()
print(f"Throughput records : {throughput3_count}")
print(f"Average Throughput : {throughput3_avg:.3f} Mbps")
print(f"Minimum Throughput : {throughput3_min:.3f} Mbps")
print(f"Maximum Throughput : {throughput3_max:.3f} Mbps")

print()
print(f"RTT records        : {rtt3_count}")
print(f"Average RTT        : {rtt3_avg:.3f} ms")
print(f"Minimum RTT        : {rtt3_min:.3f} ms")
print(f"Maximum RTT        : {rtt3_max:.3f} ms")
print(f"RTT Packet Loss    : {rtt3_loss:.3f}%")

print()
print(f"Latency records    : {latency3_count}")
print(f"Average Latency    : {latency3_avg:.3f} ms")
print(f"Average Jitter     : {jitter3_avg:.3f} ms")
print(f"Latency Packet Loss: {latency3_loss:.3f}%")

print()
print("Task 1 completed successfully.")

{"took":4126,"timed_out":false,"_shards":{"total":60,"successful":60,"skipped":0,"failed":0},"hits":{"total":{"value":6719,"relation":"eq"},"max_score":null,"hits":[]},"aggregations":{"min_value":{"value":1.56042768E8},"avg_value":{"value":3.9235831444679266E8},"max_value":{"value":4.9443248E8}}}{"took":14,"timed_out":false,"_shards":{"total":60,"successful":60,"skipped":0,"failed":0},"hits":{"total":{"value":7628,"relation":"eq"},"max_score":null,"hits":[]},"aggregations":{"avg_loss":{"value":5.2438385679137974E-5},"max_rtt":{"value":0.15301699936389923},"avg_rtt":{"value":0.05962840682476775},"min_rtt":{"value":0.05945799872279167}}}{"took":570,"timed_out":false,"_shards":{"total":60,"successful":60,"skipped":0,"failed":0},"hits":{"total":{"value":2352898,"relation":"eq"},"max_score":null,"hits":[]},"aggregations":{"avg_loss":{"value":6.399958097842493E-6},"avg_latency":{"value":34.12645705262257},"avg_jitter":{"value":0.11640186865477598}}}
FULL ARCHIVE AUTOMATIC SITE-TO-SITE PERFOR

In [6]:
# ============================================================
# CONTRIBUTION TASK 2
# Automatic Site-to-Site Path Quality Score
# Notebook 3: remote-DALL -> central-SEAT
# ============================================================

# SAME project-defined thresholds used in Notebook 2

THROUGHPUT_TARGET = 500.0   # Mbps
RTT_LIMIT = 200.0           # ms
LOSS_LIMIT = 5.0            # %
JITTER_LIMIT = 100.0        # ms


# ------------------------------------------------------------
# Normalize metrics to 0-100
# ------------------------------------------------------------

throughput3_score = min(
    100,
    (throughput3_avg / THROUGHPUT_TARGET) * 100
)

rtt3_score = max(
    0,
    100 - (rtt3_avg / RTT_LIMIT) * 100
)

loss3_score = max(
    0,
    100 - (latency3_loss / LOSS_LIMIT) * 100
)

jitter3_score = max(
    0,
    100 - (jitter3_avg / JITTER_LIMIT) * 100
)


# ------------------------------------------------------------
# Weighted path quality score
# ------------------------------------------------------------

quality3_score = (
    throughput3_score * 0.40 +
    rtt3_score        * 0.30 +
    loss3_score       * 0.20 +
    jitter3_score     * 0.10
)


# ------------------------------------------------------------
# Quality classification
# ------------------------------------------------------------

if quality3_score >= 90:
    quality3_level = "Excellent"

elif quality3_score >= 75:
    quality3_level = "Good"

elif quality3_score >= 60:
    quality3_level = "Moderate"

else:
    quality3_level = "Poor"


# ------------------------------------------------------------
# Display result
# ------------------------------------------------------------

print("=" * 64)
print("AUTOMATIC SITE-TO-SITE PATH QUALITY SCORE")
print("Notebook 3: remote-DALL -> central-SEAT")
print("=" * 64)

print()
print("Historical Average Metrics")
print("--------------------------")

print(f"Throughput       : {throughput3_avg:.3f} Mbps")
print(f"RTT              : {rtt3_avg:.3f} ms")
print(f"Packet Loss      : {latency3_loss:.3f}%")
print(f"Jitter           : {jitter3_avg:.3f} ms")

print()
print("Normalized Metric Scores")
print("------------------------")

print(f"Throughput Score : {throughput3_score:.2f}/100")
print(f"RTT Score        : {rtt3_score:.2f}/100")
print(f"Loss Score       : {loss3_score:.2f}/100")
print(f"Jitter Score     : {jitter3_score:.2f}/100")

print()
print("=" * 64)

print(f"FINAL QUALITY SCORE : {quality3_score:.2f}/100")
print(f"QUALITY LEVEL       : {quality3_level}")

print("=" * 64)

print()
print("Project-defined score weights:")
print("Throughput  : 40%")
print("RTT         : 30%")
print("Packet Loss : 20%")
print("Jitter      : 10%")

print()
print("Task 2 completed successfully.")

AUTOMATIC SITE-TO-SITE PATH QUALITY SCORE
Notebook 3: remote-DALL -> central-SEAT

Historical Average Metrics
--------------------------
Throughput       : 392.358 Mbps
RTT              : 59.628 ms
Packet Loss      : 0.001%
Jitter           : 0.116 ms

Normalized Metric Scores
------------------------
Throughput Score : 78.47/100
RTT Score        : 70.19/100
Loss Score       : 99.99/100
Jitter Score     : 99.88/100

FINAL QUALITY SCORE : 82.43/100
QUALITY LEVEL       : Good

Project-defined score weights:
Throughput  : 40%
RTT         : 30%
Packet Loss : 20%
Jitter      : 10%

Task 2 completed successfully.


In [7]:
# ============================================================
# CONTRIBUTION TASK 3
# AUTOMATIC DEGRADED-PATH DETECTION
#
# Notebook 3: remote-DALL -> central-SEAT
#
# Archive-derived rules:
#   Low Throughput = mean - 2 standard deviations
#   High RTT       = mean + 2 standard deviations
#   High Jitter    = mean + 2 standard deviations
#   Packet Loss    = any value > 0
#
# Thresholds are derived automatically from historical data.
# These are project-defined analytical rules and are not
# official perfSONAR thresholds.
# ============================================================


# ============================================================
# 1. THROUGHPUT BASELINE STATISTICS
# ============================================================

throughput3_stats_query = {
    "size": 0,

    "query": {
        "bool": {
            "must": [
                {"match": {"test.type": "throughput"}},
                {"term": {"result.succeeded": True}}
            ]
        }
    },

    "aggs": {
        "stats": {
            "extended_stats": {
                "field": "result.throughput"
            }
        }
    }
}

throughput3_stats_data = query_opensearch_nb3(
    throughput3_stats_query
)

throughput3_stats = (
    throughput3_stats_data["aggregations"]["stats"]
)

throughput3_mean_bps = throughput3_stats["avg"]
throughput3_std_bps = throughput3_stats["std_deviation"]

throughput3_threshold_bps = (
    throughput3_mean_bps
    - 2 * throughput3_std_bps
)

throughput3_threshold_mbps = (
    throughput3_threshold_bps / 1_000_000
)


# ============================================================
# 2. RTT BASELINE STATISTICS
# ============================================================

rtt3_stats_query = {
    "size": 0,

    "query": {
        "bool": {
            "must": [
                {"match": {"test.type": "rtt"}},
                {"term": {"result.succeeded": True}}
            ]
        }
    },

    "aggs": {
        "stats": {
            "extended_stats": {
                "field": "result.rtt.mean"
            }
        }
    }
}

rtt3_stats_data = query_opensearch_nb3(
    rtt3_stats_query
)

rtt3_stats = (
    rtt3_stats_data["aggregations"]["stats"]
)

rtt3_mean_seconds = rtt3_stats["avg"]
rtt3_std_seconds = rtt3_stats["std_deviation"]

rtt3_threshold_seconds = (
    rtt3_mean_seconds
    + 2 * rtt3_std_seconds
)

rtt3_threshold_ms = (
    rtt3_threshold_seconds * 1000
)


# ============================================================
# 3. JITTER BASELINE STATISTICS
# ============================================================

jitter3_stats_query = {
    "size": 0,

    "query": {
        "bool": {
            "must": [
                {"match": {"test.type": "latencybg"}},
                {"term": {"result.succeeded": True}}
            ]
        }
    },

    "aggs": {
        "stats": {
            "extended_stats": {
                "field": "result.latency.stddev"
            }
        }
    }
}

jitter3_stats_data = query_opensearch_nb3(
    jitter3_stats_query
)

jitter3_stats = (
    jitter3_stats_data["aggregations"]["stats"]
)

jitter3_mean = jitter3_stats["avg"]
jitter3_std = jitter3_stats["std_deviation"]

jitter3_threshold = (
    jitter3_mean
    + 2 * jitter3_std
)


# ============================================================
# 4. COUNT LOW-THROUGHPUT OBSERVATIONS
# ============================================================

low_throughput3_query = {
    "size": 0,
    "track_total_hits": True,

    "query": {
        "bool": {
            "must": [
                {"match": {"test.type": "throughput"}},
                {"term": {"result.succeeded": True}},

                {
                    "range": {
                        "result.throughput": {
                            "lt": throughput3_threshold_bps
                        }
                    }
                }
            ]
        }
    }
}

low_throughput3_data = query_opensearch_nb3(
    low_throughput3_query
)

low_throughput3_count = (
    low_throughput3_data["hits"]["total"]["value"]
)


# ============================================================
# 5. COUNT HIGH-RTT OBSERVATIONS
# ============================================================

high_rtt3_query = {
    "size": 0,
    "track_total_hits": True,

    "query": {
        "bool": {
            "must": [
                {"match": {"test.type": "rtt"}},
                {"term": {"result.succeeded": True}},

                {
                    "range": {
                        "result.rtt.mean": {
                            "gt": rtt3_threshold_seconds
                        }
                    }
                }
            ]
        }
    }
}

high_rtt3_data = query_opensearch_nb3(
    high_rtt3_query
)

high_rtt3_count = (
    high_rtt3_data["hits"]["total"]["value"]
)


# ============================================================
# 6. COUNT HIGH-JITTER OBSERVATIONS
# ============================================================

high_jitter3_query = {
    "size": 0,
    "track_total_hits": True,

    "query": {
        "bool": {
            "must": [
                {"match": {"test.type": "latencybg"}},
                {"term": {"result.succeeded": True}},

                {
                    "range": {
                        "result.latency.stddev": {
                            "gt": jitter3_threshold
                        }
                    }
                }
            ]
        }
    }
}

high_jitter3_data = query_opensearch_nb3(
    high_jitter3_query
)

high_jitter3_count = (
    high_jitter3_data["hits"]["total"]["value"]
)


# ============================================================
# 7. COUNT PACKET-LOSS OBSERVATIONS
# ============================================================

packet_loss3_query = {
    "size": 0,
    "track_total_hits": True,

    "query": {
        "bool": {
            "must": [
                {"match": {"test.type": "latencybg"}},
                {"term": {"result.succeeded": True}},

                {
                    "range": {
                        "result.packets.loss": {
                            "gt": 0
                        }
                    }
                }
            ]
        }
    }
}

packet_loss3_data = query_opensearch_nb3(
    packet_loss3_query
)

packet_loss3_count = (
    packet_loss3_data["hits"]["total"]["value"]
)


# ============================================================
# 8. CALCULATE ABNORMAL-OBSERVATION RATES
# ============================================================

throughput3_degradation_rate = (
    low_throughput3_count
    / throughput3_count * 100
    if throughput3_count > 0
    else 0
)

rtt3_degradation_rate = (
    high_rtt3_count
    / rtt3_count * 100
    if rtt3_count > 0
    else 0
)

jitter3_degradation_rate = (
    high_jitter3_count
    / latency3_count * 100
    if latency3_count > 0
    else 0
)

loss3_observation_rate = (
    packet_loss3_count
    / latency3_count * 100
    if latency3_count > 0
    else 0
)


# ============================================================
# 9. DETERMINE PATH STATUS
# ============================================================

detected3_conditions = []

if low_throughput3_count > 0:
    detected3_conditions.append(
        "Low-throughput observations"
    )

if high_rtt3_count > 0:
    detected3_conditions.append(
        "High-RTT observations"
    )

if high_jitter3_count > 0:
    detected3_conditions.append(
        "High-jitter observations"
    )

if packet_loss3_count > 0:
    detected3_conditions.append(
        "Packet-loss observations"
    )


if len(detected3_conditions) == 0:

    path3_status = "NO DEGRADATION DETECTED"

else:

    path3_status = "DEGRADATION EVENTS DETECTED"


# ============================================================
# 10. DISPLAY ARCHIVE-DERIVED THRESHOLDS
# ============================================================

print()
print("=" * 74)
print("          ARCHIVE-DERIVED DEGRADATION THRESHOLDS")
print("          Notebook 3: remote-DALL -> central-SEAT")
print("=" * 74)

print()

print(
    f"Throughput historical mean : "
    f"{throughput3_mean_bps / 1_000_000:.3f} Mbps"
)

print(
    f"Throughput standard dev.   : "
    f"{throughput3_std_bps / 1_000_000:.3f} Mbps"
)

print(
    f"Low throughput threshold   : "
    f"< {throughput3_threshold_mbps:.3f} Mbps"
)

print()

print(
    f"RTT historical mean        : "
    f"{rtt3_mean_seconds * 1000:.3f} ms"
)

print(
    f"RTT standard deviation     : "
    f"{rtt3_std_seconds * 1000:.3f} ms"
)

print(
    f"High RTT threshold         : "
    f"> {rtt3_threshold_ms:.3f} ms"
)

print()

print(
    f"Jitter historical mean     : "
    f"{jitter3_mean:.3f} ms"
)

print(
    f"Jitter standard deviation  : "
    f"{jitter3_std:.3f} ms"
)

print(
    f"High jitter threshold      : "
    f"> {jitter3_threshold:.3f} ms"
)

print()

print(
    "Packet-loss threshold      : > 0"
)


# ============================================================
# 11. DISPLAY DETECTED ABNORMAL OBSERVATIONS
# ============================================================

print()
print("=" * 74)
print("              AUTOMATIC DEGRADED-PATH DETECTION")
print("              Notebook 3: remote-DALL -> central-SEAT")
print("=" * 74)

print()

print(
    f"Low-throughput observations : "
    f"{low_throughput3_count} / {throughput3_count} "
    f"({throughput3_degradation_rate:.3f}%)"
)

print(
    f"High-RTT observations       : "
    f"{high_rtt3_count} / {rtt3_count} "
    f"({rtt3_degradation_rate:.3f}%)"
)

print(
    f"High-jitter observations    : "
    f"{high_jitter3_count} / {latency3_count} "
    f"({jitter3_degradation_rate:.3f}%)"
)

print(
    f"Packet-loss observations    : "
    f"{packet_loss3_count} / {latency3_count} "
    f"({loss3_observation_rate:.3f}%)"
)


# ============================================================
# 12. FINAL DETECTOR RESULT
# ============================================================

print()
print("=" * 74)
print("PATH STATUS")
print("=" * 74)

print()
print(path3_status)

if detected3_conditions:

    print()
    print("Detected conditions:")

    for condition in detected3_conditions:
        print(f"  - {condition}")

else:

    print()
    print(
        "No abnormal conditions were detected "
        "using the archive-derived rules."
    )


print()

print(
    "Note: thresholds are derived from the historical archive "
    "using mean +/- 2 standard deviations."
)

print(
    "These are project-defined analytical rules and are not "
    "official perfSONAR thresholds."
)

print()

print(
    "Task 3: Automatic degraded-path detection completed."
)

{"took":963,"timed_out":false,"_shards":{"total":60,"successful":60,"skipped":0,"failed":0},"hits":{"total":{"value":6719,"relation":"eq"},"max_score":null,"hits":[]},"aggregations":{"stats":{"count":6719,"min":1.56042768E8,"max":4.9443248E8,"avg":3.9235831444679266E8,"sum":2.636255514768E12,"sum_of_squares":1.0398761559442412E21,"variance":8.214594015191351E14,"variance_population":8.214594015191351E14,"variance_sampling":8.215816788935798E14,"std_deviation":2.8661113054435536E7,"std_deviation_population":2.8661113054435536E7,"std_deviation_sampling":2.8663246133220498E7,"std_deviation_bounds":{"upper":4.496805405556637E8,"lower":3.350360883379216E8,"upper_population":4.496805405556637E8,"lower_population":3.350360883379216E8,"upper_sampling":4.4968480671323365E8,"lower_sampling":3.350318221803517E8}}}}{"took":9,"timed_out":false,"_shards":{"total":60,"successful":60,"skipped":0,"failed":0},"hits":{"total":{"value":7629,"relation":"eq"},"max_score":null,"hits":[]},"aggregations":{"sta

In [8]:
# ============================================================
# CONTRIBUTION TASK 4
# AUTOMATED EXPERIMENT SUMMARY
#
# Notebook 3: remote-DALL -> central-SEAT
#
# Combines:
#   Task 1 -> Full-archive performance analysis
#   Task 2 -> Path-quality score
#   Task 3 -> Automatic degradation detection
#
# No experiment-result values are hard-coded.
# ============================================================


# ------------------------------------------------------------
# 1. Collect Task 3 degradation results
# ------------------------------------------------------------

degradation3_results = {

    "Low throughput": {
        "count": low_throughput3_count,
        "total": throughput3_count,
        "rate": throughput3_degradation_rate
    },

    "High RTT": {
        "count": high_rtt3_count,
        "total": rtt3_count,
        "rate": rtt3_degradation_rate
    },

    "High jitter": {
        "count": high_jitter3_count,
        "total": latency3_count,
        "rate": jitter3_degradation_rate
    },

    "Packet loss": {
        "count": packet_loss3_count,
        "total": latency3_count,
        "rate": loss3_observation_rate
    }
}


# ------------------------------------------------------------
# 2. Find highest abnormal-observation rate
# ------------------------------------------------------------

dominant3_condition = max(
    degradation3_results,
    key=lambda key: degradation3_results[key]["rate"]
)

dominant3_rate = (
    degradation3_results[
        dominant3_condition
    ]["rate"]
)


# ------------------------------------------------------------
# 3. Find active degradation categories
# ------------------------------------------------------------

active3_conditions = [
    condition
    for condition, result
    in degradation3_results.items()
    if result["count"] > 0
]


# ------------------------------------------------------------
# 4. Determine overall experiment status
# ------------------------------------------------------------

if len(active3_conditions) == 0:

    experiment3_status = (
        "NO DEGRADATION OBSERVATIONS DETECTED"
    )

else:

    experiment3_status = (
        "DEGRADATION OBSERVATIONS DETECTED"
    )


# ------------------------------------------------------------
# 5. Determine main score constraints
# ------------------------------------------------------------

metric3_scores = {
    "throughput": throughput3_score,
    "RTT": rtt3_score,
    "packet loss": loss3_score,
    "jitter": jitter3_score
}

sorted3_scores = sorted(
    metric3_scores.items(),
    key=lambda item: item[1]
)

lowest3_metric = sorted3_scores[0][0]
second_lowest3_metric = sorted3_scores[1][0]


# ============================================================
# 6. FULL-ARCHIVE PERFORMANCE
# ============================================================

print()
print("=" * 76)
print("                  AUTOMATED EXPERIMENT SUMMARY")
print("                  Notebook 3: remote-DALL -> central-SEAT")
print("=" * 76)

print()

print("FULL-ARCHIVE PERFORMANCE")
print("-" * 76)

print(
    f"Throughput measurements : "
    f"{throughput3_count}"
)

print(
    f"Average throughput      : "
    f"{throughput3_avg:.3f} Mbps"
)

print(
    f"Minimum throughput      : "
    f"{throughput3_min:.3f} Mbps"
)

print(
    f"Maximum throughput      : "
    f"{throughput3_max:.3f} Mbps"
)

print()

print(
    f"RTT measurements        : "
    f"{rtt3_count}"
)

print(
    f"Average RTT             : "
    f"{rtt3_avg:.3f} ms"
)

print(
    f"Minimum RTT             : "
    f"{rtt3_min:.3f} ms"
)

print(
    f"Maximum RTT             : "
    f"{rtt3_max:.3f} ms"
)

print(
    f"RTT packet loss         : "
    f"{rtt3_loss:.3f}%"
)

print()

print(
    f"Latency measurements    : "
    f"{latency3_count}"
)

print(
    f"Average latency         : "
    f"{latency3_avg:.3f} ms"
)

print(
    f"Average jitter          : "
    f"{jitter3_avg:.3f} ms"
)

print(
    f"Latency packet loss     : "
    f"{latency3_loss:.3f}%"
)


# ============================================================
# 7. PATH QUALITY
# ============================================================

print()
print("=" * 76)
print("PATH QUALITY")
print("=" * 76)

print()

print(
    f"Quality score : "
    f"{quality3_score:.2f}/100"
)

print(
    f"Quality level : "
    f"{quality3_level}"
)

print()

print("Metric scores:")

print(
    f"  Throughput : "
    f"{throughput3_score:.2f}/100"
)

print(
    f"  RTT        : "
    f"{rtt3_score:.2f}/100"
)

print(
    f"  Loss       : "
    f"{loss3_score:.2f}/100"
)

print(
    f"  Jitter     : "
    f"{jitter3_score:.2f}/100"
)


# ============================================================
# 8. AUTOMATIC DEGRADATION ANALYSIS
# ============================================================

print()
print("=" * 76)
print("AUTOMATIC DEGRADATION ANALYSIS")
print("=" * 76)

print()

for condition, result in degradation3_results.items():

    print(
        f"{condition:<16}: "
        f"{result['count']:>7} / "
        f"{result['total']:<7} "
        f"({result['rate']:.3f}%)"
    )


# ============================================================
# 9. HIGHEST ABNORMAL RATE
# ============================================================

print()

if active3_conditions:

    print(
        f"Highest abnormal-observation category : "
        f"{dominant3_condition}"
    )

    print(
        f"Highest abnormal-observation rate     : "
        f"{dominant3_rate:.3f}%"
    )

else:

    print(
        "No abnormal observation category "
        "contained detected events."
    )


# ============================================================
# 10. OVERALL INTERPRETATION
# ============================================================

print()
print("=" * 76)
print("OVERALL EXPERIMENT INTERPRETATION")
print("=" * 76)

print()

print(
    f"Experiment status : "
    f"{experiment3_status}"
)

print(
    f"Archive-derived quality level : "
    f"{quality3_level}"
)

print()

if active3_conditions:

    print(
        "The archive contains measurable abnormal observations, "
        "but the detected conditions represent only a fraction "
        "of their corresponding measurements."
    )

    print(
        f"The highest abnormal-observation rate is "
        f"{dominant3_rate:.3f}% for "
        f"{dominant3_condition.lower()}."
    )

else:

    print(
        "No degradation observations were detected using "
        "the archive-derived rules."
    )


# ============================================================
# 11. EXPLAIN QUALITY SCORE
# ============================================================

print()

print(
    f"The two lowest normalized quality components are "
    f"{lowest3_metric} and {second_lowest3_metric}; "
    f"packet-loss and jitter performance remain comparatively strong."
)


# ============================================================
# 12. METHODOLOGICAL NOTE
# ============================================================

print()

print(
    "Note: quality scores and degradation thresholds are "
    "project-defined analytical methods and are not official "
    "perfSONAR metrics or classifications."
)

print()

print(
    "Task 4: Automated experiment summary completed."
)


                  AUTOMATED EXPERIMENT SUMMARY
                  Notebook 3: remote-DALL -> central-SEAT

FULL-ARCHIVE PERFORMANCE
----------------------------------------------------------------------------
Throughput measurements : 6719
Average throughput      : 392.358 Mbps
Minimum throughput      : 156.043 Mbps
Maximum throughput      : 494.432 Mbps

RTT measurements        : 7628
Average RTT             : 59.628 ms
Minimum RTT             : 59.458 ms
Maximum RTT             : 153.017 ms
RTT packet loss         : 0.005%

Latency measurements    : 2352898
Average latency         : 34.126 ms
Average jitter          : 0.116 ms
Latency packet loss     : 0.001%

PATH QUALITY

Quality score : 82.43/100
Quality level : Good

Metric scores:
  Throughput : 78.47/100
  RTT        : 70.19/100
  Loss       : 99.99/100
  Jitter     : 99.88/100

AUTOMATIC DEGRADATION ANALYSIS

Low throughput  :     269 / 6719    (4.004%)
High RTT        :      10 / 7628    (0.131%)
High jitter     :    9864 / 2

In [6]:
import json
import base64
import os

DALL_SOURCE = "2001:400:a100:3000:f816:3eff:febc:fd55"
DALL_TARGET = "2620:0:860:201:208:80:152:244"

query_body = {
    "size": 200,
    "query": {
        "bool": {
            "must": [
                {"match": {"test.type": "rtt"}}
            ]
        }
    },
    "sort": [
        {"pscheduler.start_time": {"order": "desc"}}
    ]
}

body_json = json.dumps(query_body)

body_b64 = base64.b64encode(
    body_json.encode("utf-8")
).decode("utf-8")

cmd = f"""
sudo bash -c '
read -r OS_USER OS_PASS < /etc/perfsonar/opensearch/opensearch_login

echo "{body_b64}" | base64 -d | \
curl -ks \
  -u "${{OS_USER}}:${{OS_PASS}}" \
  -H "Content-Type: application/json" \
  --data-binary @- \
  https://localhost/opensearch/pscheduler/_search
'
"""

stdout, stderr = ps_dall.execute(cmd)

data = json.loads(stdout)

if "error" in data:
    print(json.dumps(data["error"], indent=2))
else:
    hits = data["hits"]["hits"]

    research_hits = []

    for hit in hits:
        src = hit["_source"]
        spec = src.get("test", {}).get("spec", {})

        if (
            spec.get("source") == DALL_SOURCE
            and spec.get("dest") == DALL_TARGET
        ):
            research_hits.append(src)

    print("Matching Dallas research RTT records:", len(research_hits))

    if research_hits:
        print("\n===== SAMPLE RECORD =====")
        print(json.dumps(research_hits[0], indent=2))

        with open("Dallas_rtt_sample.json", "w") as f:
            json.dump(research_hits, f, indent=2)

        print("\nSaved: Dallas_rtt_sample.json")

if stderr:
    print("\nSTDERR:")
    print(stderr)

{"took":1071,"timed_out":false,"_shards":{"total":54,"successful":49,"skipped":0,"failed":5,"failures":[{"shard":0,"index":".ds-pscheduler_clock-000005","node":"5JoT7w2kSK6uK699W1q3Qw","reason":{"type":"query_shard_exception","reason":"No mapping found for [pscheduler.start_time] in order to sort on","index":".ds-pscheduler_clock-000005","index_uuid":"HhcWVEAEQYK7vMjL_mKOCw"}},{"shard":0,"index":".ds-pscheduler_mtu-000005","node":"5JoT7w2kSK6uK699W1q3Qw","reason":{"type":"query_shard_exception","reason":"No mapping found for [pscheduler.start_time] in order to sort on","index":".ds-pscheduler_mtu-000005","index_uuid":"XxsJ42JkQfi5smFIgdIJCQ"}},{"shard":0,"index":".ds-pscheduler_rtt-000005","node":"5JoT7w2kSK6uK699W1q3Qw","reason":{"type":"query_shard_exception","reason":"No mapping found for [pscheduler.start_time] in order to sort on","index":".ds-pscheduler_rtt-000005","index_uuid":"u638MxatTIOeQvGxlv4mOA"}},{"shard":0,"index":".ds-pscheduler_throughput-000005","node":"5JoT7w2kSK6uK6

In [10]:
cmd = r'''
sudo bash -c '
read -r OS_USER OS_PASS < /etc/perfsonar/opensearch/opensearch_login

curl -ks -u "${OS_USER}:${OS_PASS}" \
  -H "Content-Type: application/json" \
  "https://localhost/opensearch/pscheduler/_search" \
  -d '"'"'
{
  "size": 20,
  "sort": [
    {
      "pscheduler.start_time": {
        "order": "desc",
        "unmapped_type": "date"
      }
    }
  ],
  "query": {
    "bool": {
      "must": [
        {
          "match_phrase": {
            "test.type": "trace"
          }
        },
        {
          "match_phrase": {
            "test.spec.source": "2001:400:a100:3000:f816:3eff:febc:fd55"
          }
        },
        {
          "match_phrase": {
            "test.spec.dest": "2620:0:860:201:208:80:152:244"
          }
        }
      ]
    }
  }
}
'"'"'
'
'''

stdout, stderr = ps_dall.execute(cmd)

print(stdout)
if stderr:
    print("STDERR:", stderr)

{"took":763,"timed_out":false,"_shards":{"total":54,"successful":54,"skipped":0,"failed":0},"hits":{"total":{"value":169,"relation":"eq"},"max_score":null,"hits":[{"_index":".ds-pscheduler_trace-000009","_id":"76jy0aEBnKHAimkwNZdC","_score":null,"_source":{"@version":"1","@timestamp":"2026-11-12T22:47:53.723264567Z","id":"18b35906-04ae-461e-b54d-a5dfd6935485","schedule":{"sliprand":true,"slip":900.0,"until":"2026-11-13T04:41:46Z","start":"2026-11-12T04:41:46Z","repeat":900.0},"meta":{"destination":{"hostname":"ripe-atlas-codfw.wikimedia.org","as":{"organization":"Wikimedia Foundation Inc."},"ip":"2620:0:860:201:208:80:152:244","geo":{"continent_name":"North America","as":{"number":14907},"country_name":"United States","location":{"lat":37.751,"lon":-97.822}}},"observer":{"hostname":"remote-DALL","as":{"organization":"ESnet"},"ip":"2001:400:a100:3000:f816:3eff:febc:fd55","geo":{"continent_name":"North America","as":{"number":293},"country_name":"United States","location":{"lat":37.751,"

In [12]:
import json
from pathlib import Path

trace_response = json.loads(stdout)

records = [
    hit["_source"]
    for hit in trace_response["hits"]["hits"]
]

print("Records:", len(records))

PROJECT_ROOT = Path("../network_measurement_case_study").resolve()

raw_dir = (
    PROJECT_ROOT
    / "cross_tool_analysis"
    / "raw_perfsonar"
    / "traceroute"
    / "raw"
)

raw_dir.mkdir(
    parents=True,
    exist_ok=True
)

output_file = (
    raw_dir
    / "perfsonar_ripe_dallas_traceroute_test20.json"
)

with open(output_file, "w") as f:
    json.dump(records, f, indent=2)

print("Saved:", output_file)

Records: 20
Saved: /home/fabric/work/network_measurement_case_study/cross_tool_analysis/raw_perfsonar/traceroute/raw/perfsonar_ripe_dallas_traceroute_test20.json


In [15]:
import json

DALL_SOURCE = "2001:400:a100:3000:f816:3eff:febc:fd55"

PERFSONAR_TRACE_TARGETS = {
    "RIPE-Dallas": "2620:0:860:201:208:80:152:244",
    "RIPE-Seattle": "2604:e180:de08::5",
    "RIPE-NYC": "2604:a880:800:10::a0:b001",
    "Google": "2001:4860:4860::8888",
    "Cloudflare": "2606:4700:4700::1111",
    "FABRIC-SEAT": "2001:400:a100:3080:f816:3eff:fe23:6116"
}

availability = {}


def run_query(query):
    query_json = json.dumps(query)

    cmd = f"""
sudo bash -c '
read -r OS_USER OS_PASS < /etc/perfsonar/opensearch/opensearch_login

curl -ks -u "${{OS_USER}}:${{OS_PASS}}" \
  -H "Content-Type: application/json" \
  "https://localhost/opensearch/pscheduler/_search" \
  -d '"'"'{query_json}'"'"'
'
"""

    out, err = ps_dall.execute(cmd)

    if err:
        print("STDERR:", err)

    return json.loads(out)


for target_name, target_ip in PERFSONAR_TRACE_TARGETS.items():

    print(f"\nChecking {target_name}...")

    common_filters = [
        {
            "match_phrase": {
                "test.type": "trace"
            }
        },
        {
            "match_phrase": {
                "test.spec.source": DALL_SOURCE
            }
        },
        {
            "match_phrase": {
                "test.spec.dest": target_ip
            }
        },
        {
            "match_phrase": {
                "reference.display-task-group": "Research 15m"
            }
        }
    ]

    # Earliest research traceroute record
    earliest_query = {
        "size": 1,
        "sort": [
            {
                "pscheduler.start_time": {
                    "order": "asc",
                    "unmapped_type": "date"
                }
            }
        ],
        "query": {
            "bool": {
                "must": common_filters
            }
        }
    }

    # Latest research traceroute record
    latest_query = {
        "size": 1,
        "sort": [
            {
                "pscheduler.start_time": {
                    "order": "desc",
                    "unmapped_type": "date"
                }
            }
        ],
        "query": {
            "bool": {
                "must": common_filters
            }
        }
    }

    earliest_resp = run_query(earliest_query)
    latest_resp = run_query(latest_query)

    earliest_hits = earliest_resp["hits"]["hits"]
    latest_hits = latest_resp["hits"]["hits"]

    if not earliest_hits or not latest_hits:

        availability[target_name] = {
            "first": None,
            "last": None
        }

        print("No Research 15m records found.")
        continue

    first_time = (
        earliest_hits[0]
        ["_source"]
        ["pscheduler"]
        ["start_time"]
    )

    last_time = (
        latest_hits[0]
        ["_source"]
        ["pscheduler"]
        ["start_time"]
    )

    availability[target_name] = {
        "first": first_time,
        "last": last_time
    }

    print("First:", first_time)
    print("Last: ", last_time)


print("\n===== perfSONAR Research 15m availability =====")

for target, times in availability.items():

    print(
        f"{target:12} | "
        f"First: {times['first']} | "
        f"Last: {times['last']}"
    )


Checking RIPE-Dallas...
{"took":641,"timed_out":false,"_shards":{"total":54,"successful":54,"skipped":0,"failed":0},"hits":{"total":{"value":172,"relation":"eq"},"max_score":null,"hits":[{"_index":".ds-pscheduler_trace-000008","_id":"huTtyKEBnKHAimkwv5Xq","_score":null,"_source":{"@version":"1","@timestamp":"2026-11-11T04:46:26.563040936Z","id":"611aad5d-9619-46b1-875a-ba3074f35f6b","schedule":{"sliprand":true,"slip":900.0,"until":"2026-11-12T04:41:46Z","repeat":900.0},"meta":{"destination":{"hostname":"ripe-atlas-codfw.wikimedia.org","as":{"organization":"Wikimedia Foundation Inc."},"ip":"2620:0:860:201:208:80:152:244","geo":{"continent_name":"North America","as":{"number":14907},"country_name":"United States","location":{"lat":37.751,"lon":-97.822}}},"observer":{"hostname":"remote-DALL","as":{"organization":"ESnet"},"ip":"2001:400:a100:3000:f816:3eff:febc:fd55","geo":{"continent_name":"North America","as":{"number":293},"country_name":"United States","location":{"lat":37.751,"lon":-

In [16]:
import json
from pathlib import Path

COMMON_START = "2026-11-11T21:15:00Z"
COMMON_END   = "2026-11-12T18:26:00Z"

DALL_SOURCE = "2001:400:a100:3000:f816:3eff:febc:fd55"

PERFSONAR_TRACE_TARGETS = {
    "RIPE-Dallas": "2620:0:860:201:208:80:152:244",
    "RIPE-Seattle": "2604:e180:de08::5",
    "RIPE-NYC": "2604:a880:800:10::a0:b001",
    "Google": "2001:4860:4860::8888",
    "Cloudflare": "2606:4700:4700::1111",
    "FABRIC-SEAT": "2001:400:a100:3080:f816:3eff:fe23:6116"
}

PROJECT_ROOT = Path("../network_measurement_case_study").resolve()

RAW_DIR = (
    PROJECT_ROOT
    / "cross_tool_analysis"
    / "raw_perfsonar"
    / "traceroute"
    / "raw"
)

RAW_DIR.mkdir(parents=True, exist_ok=True)


def run_query(query):
    query_json = json.dumps(query)

    cmd = f"""
sudo bash -c '
read -r OS_USER OS_PASS < /etc/perfsonar/opensearch/opensearch_login

curl -ks -u "${{OS_USER}}:${{OS_PASS}}" \
  -H "Content-Type: application/json" \
  "https://localhost/opensearch/pscheduler/_search" \
  -d '"'"'{query_json}'"'"'
'
"""

    out, err = ps_dall.execute(cmd)

    if err:
        print("STDERR:", err)

    return json.loads(out)


for target_name, target_ip in PERFSONAR_TRACE_TARGETS.items():

    print(f"\nExtracting {target_name}...")

    query = {
        "size": 500,
        "sort": [
            {
                "pscheduler.start_time": {
                    "order": "asc",
                    "unmapped_type": "date"
                }
            }
        ],
        "query": {
            "bool": {
                "must": [
                    {"match_phrase": {"test.type": "trace"}},
                    {"match_phrase": {"test.spec.source": DALL_SOURCE}},
                    {"match_phrase": {"test.spec.dest": target_ip}},
                    {
                        "match_phrase": {
                            "reference.display-task-group": "Research 15m"
                        }
                    },
                    {
                        "range": {
                            "pscheduler.start_time": {
                                "gte": COMMON_START,
                                "lte": COMMON_END
                            }
                        }
                    }
                ]
            }
        }
    }

    response = run_query(query)

    records = [
        hit["_source"]
        for hit in response["hits"]["hits"]
    ]

    safe_name = target_name.lower().replace("-", "_")

    output_file = (
        RAW_DIR
        / f"perfsonar_{safe_name}_traceroute_common_window_raw.json"
    )

    with open(output_file, "w") as f:
        json.dump(records, f, indent=2)

    print("Records:", len(records))

    if records:
        print(
            "First:",
            records[0]["pscheduler"]["start_time"]
        )
        print(
            "Last: ",
            records[-1]["pscheduler"]["start_time"]
        )

    print("Saved:", output_file)


Extracting RIPE-Dallas...
{"took":939,"timed_out":false,"_shards":{"total":54,"successful":54,"skipped":42,"failed":0},"hits":{"total":{"value":85,"relation":"eq"},"max_score":null,"hits":[{"_index":".ds-pscheduler_trace-000008","_id":"NzF4zKEBnKHAimkwTowx","_score":null,"_source":{"@version":"1","@timestamp":"2026-11-11T21:16:38.640333100Z","id":"5fb13c88-9c6b-4171-8568-7075b5910ab5","schedule":{"sliprand":true,"slip":900.0,"until":"2026-11-12T04:41:46Z","repeat":900.0},"meta":{"destination":{"hostname":"ripe-atlas-codfw.wikimedia.org","as":{"organization":"Wikimedia Foundation Inc."},"ip":"2620:0:860:201:208:80:152:244","geo":{"continent_name":"North America","as":{"number":14907},"country_name":"United States","location":{"lat":37.751,"lon":-97.822}}},"observer":{"hostname":"remote-DALL","as":{"organization":"ESnet"},"ip":"2001:400:a100:3000:f816:3eff:febc:fd55","geo":{"continent_name":"North America","as":{"number":293},"country_name":"United States","location":{"lat":37.751,"lon"